# Introducción a Redes Neuronales Convolucionales (CNN) con FashionMNIST

En esta notebook, implementaremos una Red Neuronal Convolucional (CNN) básica y aprenderemos conceptos fundamentales como convoluciones y pooling, utilizando el dataset [FashionMNIST](https://en.wikipedia.org/wiki/Fashion_MNIST). El objetivo principal es entender cómo las CNNs pueden ser aplicadas a problemas de clasificación de imágenes.

## Objetivos

1. **Entender los conceptos básicos de las Redes Neuronales Convolucionales (CNN)**, incluyendo convoluciones, capas de pooling y capas totalmente conectadas.
2. **Entrenar una CNN** en PyTorch utilizando el dataset FashionMNIST.
3. **Evaluar el rendimiento del modelo** utilizando métricas adecuadas y visualización de resultados.

## Contenido

1. Configuración de bibliotecas y semillas para reproducibilidad.
2. Convoluciones: qué son, filtros a mano sobre una imagen, `stride`, `padding` y `bias`.
3. *(Lectura opcional)* Convoluciones con varios canales y conteo de parámetros.
4. Pooling.
5. Carga de FashionMNIST con `torchvision` y transformaciones.
6. Implementación y entrenamiento de LeNet-5.
7. Evaluación en test.
8. *(Lectura opcional)* Qué aprendió la red: filtros, feature maps y errores.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, random_split
from torchvision.transforms import v2 as T
from torchvision.io import read_image, ImageReadMode

from torchinfo import summary

from pathlib import Path

from utils import (
    get_device,
    get_num_workers,
    train,
    plot_training,
    model_classification_report,
    show_tensor_image,
    show_tensor_images,
)


In [ ]:
# Fijamos la semilla para que los resultados sean reproducibles
SEED = 34

torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True  # cuDNN tiene varias implementaciones de cada operación (convoluciones, etc.); esto fuerza las que dan siempre el mismo resultado, a costa de velocidad


In [ ]:
DEVICE = get_device()  # cuda > mps > xpu > cpu
NUM_WORKERS = get_num_workers()  # Linux: mitad de los núcleos disponibles (máx. 8); Windows/macOS: 0 (ver docstring)

print(f"Usando {DEVICE}")
print(f"Usando {NUM_WORKERS}")


In [ ]:
BATCH_SIZE = 128  # tamaño del batch


## Intro a CNNs

![Image](assets/cnn.png)

Las redes convolucionales (CNNs) son muy utilizadas en el campo de **computer vision**, es decir, en problemas relacionados con imágenes como clasificación, detección de objetos, segmentación, etc. Los dos conceptos clave en las CNNs son las **convoluciones** y el **pooling**. Debido a que los píxeles de una imagen tienen una relación espacial, las CNNs son capaces de capturar patrones locales en la imagen, como bordes, texturas, etc.

### ¿Por qué no usar el MLP de la clase pasada?

Una imagen de FashionMNIST tiene 28×28 = 784 píxeles. Podríamos aplanarla en un vector de 784 valores y entrenar el MLP de la clase 03 (784 → 512 → 2048 → 10). Eso tiene dos problemas:

1. **Cantidad de parámetros**: solo la primera capa `Linear(784, 512)` tiene 784 × 512 + 512 ≈ 400 000 pesos, y el modelo completo más de 1.4 millones. Con imágenes más grandes (por ejemplo 224×224×3 = 150 528 entradas) esto se vuelve inmanejable.
2. **Se pierde la estructura espacial**: para el MLP, el píxel (3, 4) y el píxel (3, 5) son dos entradas independientes, sin relación entre sí. Si la remera se corre un píxel a la derecha, *todas* las entradas cambian y la red tiene que volver a aprender el patrón en la nueva posición.

Las CNNs resuelven esto con tres ideas:

- **Conectividad local**: cada salida mira solo una ventana pequeña de la entrada (por ejemplo 3×3 o 5×5 píxeles), no toda la imagen.
- **Pesos compartidos**: el mismo filtro (los mismos pesos) se aplica en todas las posiciones de la imagen. Un detector de bordes sirve igual arriba a la izquierda que abajo a la derecha. Esto reduce drásticamente los parámetros y hace que la red responda igual a un patrón sin importar dónde aparezca (*equivarianza a traslaciones*).
- **Jerarquía**: al apilar capas, las primeras detectan bordes, las siguientes combinan bordes en texturas y partes, y las últimas combinan partes en objetos. Es lo que muestra la figura de arriba.

LeNet-5, la red que implementamos hoy, tiene **61 706 parámetros**, 23 veces menos que el MLP, y clasifica mejor.

> **Vocabulario de la figura:** cada rectángulo azul es un **feature map** (mapa de características): la salida de aplicar *un* filtro a la imagen. Varios filtros → varios feature maps. **Subsampling** es lo que hoy llamamos **pooling**. **Fully connected** son las capas `Linear` que ya conocemos.


### Convoluciones

En el contexto de las CNNs, las convoluciones se aplican a una imagen de entrada y un filtro (kernel) para producir una imagen de salida. La operación de convolución se realiza deslizando el filtro sobre la imagen de entrada, multiplicando sus valores por los valores de los píxeles de la imagen y sumando el resultado.

![Image](https://d2l.ai/_images/correlation.svg)

> **Nota para quienes vieron convolución en Señales y Sistemas:** estrictamente, lo que hacen PyTorch y todas las librerías de deep learning es una **correlación cruzada** (el kernel se desliza tal cual, sin darlo vuelta). Por eso la figura de [Dive into Deep Learning](https://d2l.ai/chapter_convolutional-neural-networks/conv-layer.html) se llama `correlation`. Como los filtros se *aprenden*, da lo mismo: si hiciera falta el kernel invertido, la red aprendería el kernel invertido. Por costumbre se le sigue diciendo convolución.

Vamos a ver algunos ejemplos de convoluciones y cómo afectan a las imágenes. Para ello vamos a utilizar la función [`torch.nn.functional.conv2d`](https://pytorch.org/docs/stable/generated/torch.nn.functional.conv2d.html) de PyTorch.

**Vocabulario:** *filtro* y *kernel* son sinónimos. PyTorch trabaja con tensores de 4 dimensiones en el orden `[N, C, H, W]`:

- `N`: cantidad de imágenes del batch.
- `C`: cantidad de **canales**. Una imagen en escala de grises tiene 1 canal; una imagen a color tiene 3 (R, G, B); la salida de una capa con 6 filtros tiene 6 canales (un feature map por filtro).
- `H`, `W`: alto y ancho en píxeles.

Por eso, en el ejemplo siguiente, la imagen 3×3 y el kernel 2×2 se reacomodan a 4 dimensiones antes de llamar a `conv2d`.


In [ ]:
input = torch.tensor([[0, 1, 2], [3, 4, 5], [6, 7, 8]], dtype=torch.float32)
kernel = torch.tensor([[0, 1], [2, 3]], dtype=torch.float32)

# para utilizar F.conv2d, necesitamos que las dimensiones de kernel sean [out_channels, in_channels, kernel_height, kernel_width]
# y las dimensiones de input sean [batch_size, in_channels, height, width]

input = input.reshape(1, 1, *input.shape)  # [1, 1, 3, 3]
kernel = kernel.reshape(1, 1, *kernel.shape)  # [1, 1, 2, 2]

F.conv2d(input, kernel)


Es exactamente el ejemplo de la figura. El valor de arriba a la izquierda sale de superponer el kernel sobre la esquina superior izquierda de la imagen:

$$0 \cdot 0 + 1 \cdot 1 + 3 \cdot 2 + 4 \cdot 3 = 19$$

Después el kernel se corre una posición a la derecha ($1 \cdot 0 + 2 \cdot 1 + 4 \cdot 2 + 5 \cdot 3 = 25$), y así hasta recorrer toda la imagen.

**¿Por qué la salida es 2×2 y no 3×3?** Porque un kernel de 2×2 solo entra en 2 posiciones horizontales y 2 verticales dentro de una imagen de 3×3. En general, la salida mide `entrada − kernel + 1`. Más adelante vamos a ver la fórmula completa (con `padding` y `stride`).


Veamos un ejemplo de cómo se aplica una convolución a una imagen real. En este caso, vamos a definir filtros que ayudan a detectar bordes.

La función [read_image](https://pytorch.org/vision/stable/generated/torchvision.io.read_image.html) es una función auxiliar que lee una imagen de un archivo y la convierte en un tensor de PyTorch con forma `[C, H, W]` y valores enteros entre 0 y 255 (`uint8`).

> **Sobre `vmin=0, vmax=255`:** le fijamos a matplotlib el rango de la escala de grises (0 = negro, 255 = blanco). Si no se lo pasamos, matplotlib estira el rango de *cada* imagen para que su mínimo sea negro y su máximo blanco, y dos imágenes con distinto brillo se verían iguales. Lo vamos a usar en todos los ejemplos para poder comparar.


In [ ]:
image = read_image(str(Path("assets") / "PrisonMike.png"), ImageReadMode.GRAY)
print("Forma:", image.shape, "| dtype:", image.dtype, "| rango:", image.min().item(), "-", image.max().item())

show_tensor_image(image, title="Original Image", vmin=0, vmax=255)


#### Detectores de bordes direccionales

Empezamos con tres filtros de 3×3 diseñados a mano. Miremos el `horizontal_edge`:

```
-1 -1 -1
 2  2  2
-1 -1 -1
```

Si el kernel cae sobre una zona de color uniforme, la fila central positiva y las dos filas negativas se cancelan: los coeficientes **suman 0**, así que la salida es 0 (negro). Solo da un valor grande cuando la fila del medio es mucho más clara que las de arriba y abajo, es decir, sobre una **línea horizontal**. El `vertical_edge` es el mismo patrón girado, y el `diagonal_edge` responde a líneas diagonales.

Los tres kernels se apilan con `torch.cat` en la dimensión 0, que es la de `out_channels`: **cada filtro produce un canal (feature map) en la salida**.


In [ ]:
image_f = image.to(torch.float32)  # conv2d trabaja con float; los valores siguen en 0-255
image_batch = image_f.unsqueeze(0)  # agregamos la dimensión de batch: [1, 1, H, W]

print("Forma de la imagen:", image_batch.shape)

horizontal_edge = torch.tensor(
    [[[[-1, -1, -1], [2, 2, 2], [-1, -1, -1]]]], dtype=torch.float32
)

vertical_edge = torch.tensor(
    [[[[-1, 2, -1], [-1, 2, -1], [-1, 2, -1]]]], dtype=torch.float32
)

diagonal_edge = torch.tensor(
    [[[[2, -1, -1], [-1, 2, -1], [-1, -1, 2]]]], dtype=torch.float32
)


# Apilar los kernels para aplicarlos todos a la vez: [3, 1, 3, 3] = 3 filtros, 1 canal de entrada, 3x3
kernels = torch.cat([horizontal_edge, vertical_edge, diagonal_edge])
print("Forma de los kernels:", kernels.shape)

# Aplicar la convolución usando los kernels definidos
output = F.conv2d(image_batch, kernels)
print("Forma de la salida:", output.shape)  # 3 canales de salida, uno por filtro

# Mostrar las imágenes resultantes
show_tensor_images(
    [output[0, i].unsqueeze(0) for i in range(output.shape[1])],
    titles=["Horizontal", "Vertical", "Diagonal"],
    figsize=(20, 5),
    vmin=0,
    vmax=255,
)


#### Otros filtros clásicos

Definimos los siguientes filtros:

- `identity`: un filtro que no modifica la imagen (un 1 en el centro y ceros alrededor).
- `edge`: detecta bordes en todas las direcciones. Sus coeficientes suman 0, como los anteriores.
- `sharpen`: resalta los bordes. Es la identidad más un detector de bordes; sus coeficientes suman 1, así que conserva el brillo general.
- `box_blur`: suaviza la imagen reemplazando cada píxel por el promedio de sus 9 vecinos. Se divide por 9 para que los coeficientes sumen 1 y la imagen no se aclare.
- `gau_blur`: también suaviza, pero dando más peso al centro (aproxima una campana de Gauss). Se divide por 16 por la misma razón.

> Estos filtros existen desde mucho antes de las redes neuronales: son los mismos que usan Photoshop o GIMP. En [Setosa – Image Kernels](https://setosa.io/ev/image-kernels/) se pueden probar de forma interactiva.


In [ ]:
identity = torch.tensor([[[[0, 0, 0], [0, 1, 0], [0, 0, 0]]]], dtype=torch.float32)

edge = torch.tensor([[[[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]]]], dtype=torch.float32)

sharpen = torch.tensor([[[[0, -1, 0], [-1, 5, -1], [0, -1, 0]]]], dtype=torch.float32)

box_blur = torch.tensor([[[[1, 1, 1], [1, 1, 1], [1, 1, 1]]]], dtype=torch.float32) / 9

gau_blur = torch.tensor([[[[1, 2, 1], [2, 4, 2], [1, 2, 1]]]], dtype=torch.float32) / 16


# Apilar los kernels para aplicarlos todos a la vez
kernels = torch.cat([identity, edge, sharpen, box_blur, gau_blur])
print("Forma de los kernels:", kernels.shape)

# Aplicar la convolución usando los kernels definidos
output = F.conv2d(image_batch, kernels)
print("Forma de la salida:", output.shape)

# Mostrar las imágenes resultantes
show_tensor_images(
    [output[0, i].unsqueeze(0) for i in range(output.shape[1])],
    titles=["Identity", "Edge Detection", "Sharpen", "Box Blur", "Gaussian Blur"],
    figsize=(20, 5),
    vmin=0,
    vmax=255,
)


> **¿Por qué "Edge Detection" se ve casi negra?** La salida de un filtro no está limitada a 0-255: el detector de bordes produce valores negativos (donde el centro es más oscuro que el entorno) y positivos mayores a 255 (bordes muy marcados). Con `vmin=0` matplotlib pinta de negro todo lo negativo y de blanco todo lo que supera 255. Veamos el rango real de la salida:


In [ ]:
edge_out = output[0, 1]
print(f"Rango de la salida de edge: [{edge_out.min().item():.0f}, {edge_out.max().item():.0f}]")

# el valor absoluto muestra los bordes sin importar el signo
show_tensor_images(
    [edge_out.unsqueeze(0), edge_out.abs().unsqueeze(0)],
    titles=["Edge (con vmin=0, los negativos se pierden)", "|Edge| (bordes en ambas direcciones)"],
    figsize=(12, 5),
    vmin=0,
    vmax=255,
)


Guardemos esta idea: **la salida de una convolución puede ser negativa o positiva**. En una CNN, después de cada convolución se aplica una función de activación (por ejemplo ReLU, que deja pasar solo los valores positivos), igual que hacíamos después de cada `Linear`.

#### Encadenar convoluciones

Podemos aplicar una convolución sobre la salida de otra. Esto es exactamente lo que hace una CNN: cada capa procesa los feature maps de la anterior. Notar dos cosas:

- El **campo receptivo** crece: un píxel de la salida de tres convoluciones 3×3 encadenadas "ve" una zona de 7×7 de la imagen original. Las capas profundas detectan patrones más grandes.
- Si encadenamos convoluciones **sin una no linealidad entre medio**, el resultado equivale a una única convolución (con un kernel más grande). Por eso en una red siempre va una activación entre capa y capa; sin ella, apilar capas no agrega capacidad. Es el mismo argumento que vimos para el MLP.


In [ ]:
# podemos combinar
output = F.conv2d(image_batch, edge)
output = F.conv2d(output, box_blur)
output = F.conv2d(output, gau_blur)

show_tensor_image(output.abs(), title="edge -> box_blur -> gau_blur", vmin=0, vmax=255)
print("Forma de la salida:", output.shape)  # cada conv 3x3 sin padding quita 2 píxeles por lado


#### De feature maps a vector: `flatten`

Las capas `Linear` esperan un vector por muestra, `[N, features]`, no un tensor `[N, C, H, W]`. Para pasar de una cosa a la otra usamos [`flatten(start_dim=1)`](https://pytorch.org/docs/stable/generated/torch.flatten.html): aplana todo salvo la dimensión de batch. Lo vamos a necesitar dentro de LeNet, entre la última convolución y la primera `Linear`.


In [ ]:
five_maps = F.conv2d(image_batch, kernels)  # los 5 filtros de antes: [1, 5, 318, 310]
flat = five_maps.flatten(start_dim=1)

print("Antes de flatten:", five_maps.shape)
print("Después de flatten:", flat.shape, "=", "5 * 318 * 310 =", 5 * 318 * 310)


La buena noticia es que no necesitamos definir estos filtros manualmente, ya que PyTorch proporciona la capa [`torch.nn.Conv2d`](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) que inicializa los pesos de los filtros de manera aleatoria y los entrena durante el proceso de aprendizaje.

Esto quiere decir que la red neuronal aprenderá automáticamente los filtros que son útiles para el problema en cuestión. Los coeficientes del kernel son parámetros que se ajustan por backpropagation, exactamente igual que los pesos de una `nn.Linear`. En la sección opcional del final de la notebook se pueden ver los filtros que aprendió nuestra red.

> **`F.conv2d` vs `nn.Conv2d`**: es la misma relación que entre `F.relu` y `nn.ReLU`, o entre `F.linear` y `nn.Linear`. La versión de `torch.nn.functional` es una función pura: le pasamos nosotros el kernel. La versión de `torch.nn` es un módulo que **guarda el kernel como parámetro entrenable** (`.weight`) y el sesgo (`.bias`). En los modelos usamos `nn.Conv2d`.


#### Otros parámetros de la convolución

Además de los filtros, las convoluciones tienen otros parámetros importantes:

- `stride`: paso de la convolución, es decir, cuántos píxeles se desplaza el filtro en cada paso. Con `stride=2` la salida tiene la mitad de alto y ancho: es una forma barata de reducir resolución (varias redes modernas la usan en lugar del pooling).
- `padding`: relleno de la imagen, es decir, cuántos píxeles **de ceros** se añaden en cada borde de la imagen. Muchas veces se utiliza para mantener el tamaño de la imagen de salida igual al de la imagen de entrada: con `stride=1`, eso se logra con `padding = (kernel_size - 1) / 2` (por ejemplo `padding=1` para un kernel 3×3, `padding=2` para uno 5×5).
- `bias`: si se incluye un término de sesgo. Es **un solo número por filtro de salida** que se suma a todos los píxeles de ese feature map (por eso en el ejemplo de abajo `bias` es un tensor de forma `[out_channels]`).

Las dimensiones de salida se pueden calcular con la siguiente fórmula (una vez para el alto y otra para el ancho):

$$
\text{output\_size} = \left\lfloor \frac{\text{input\_size} - \text{kernel\_size} + 2 \times \text{padding}}{\text{stride}} \right\rfloor + 1
$$

El piso $\lfloor \cdot \rfloor$ importa: cuando la división no es exacta, las últimas filas o columnas que no completan una ventana se descartan.

> Las animaciones de [Dumoulin & Visin, *A guide to convolution arithmetic for deep learning*](https://github.com/vdumoulin/conv_arithmetic) muestran cada combinación de `padding` y `stride`. Vale la pena tenerlas a mano.

Veamos los tres parámetros con el filtro `identity`, que no cambia los valores, para aislar el efecto de cada uno. Primero `padding`: la imagen crece 30 píxeles **de cada lado** y aparece un marco negro (los ceros).


In [ ]:
output = F.conv2d(image_batch, identity, padding=30)  # sumamos 30 pixeles de padding de cada lado
show_tensor_image(output, title="padding=30", vmin=0, vmax=255)

print(f"Shape de la imagen original: {image_batch.shape}")
print(f"Shape de la imagen con padding: {output.shape}")  # 320 + 2*30 - 3 + 1 = 378


Con `stride=3` la salida queda de aproximadamente un tercio del tamaño. Verifiquemos la fórmula: $\lfloor (320 - 3 + 2 \cdot 5) / 3 \rfloor + 1 = \lfloor 109 \rfloor + 1 = 110$ para el alto, y $\lfloor (312 - 3 + 10) / 3 \rfloor + 1 = \lfloor 106.33 \rfloor + 1 = 107$ para el ancho.

> matplotlib dibuja cada imagen ocupando toda la figura, así que la reducción no se nota a simple vista: hay que mirar el `shape`.


In [ ]:
output = F.conv2d(image_batch, identity, stride=3, padding=5)
show_tensor_image(output, title="stride=3, padding=5", vmin=0, vmax=255)

print(f"Shape de la imagen con stride: {output.shape}")


El `bias` se suma a cada píxel de la salida. Con `bias=-100` toda la imagen se oscurece: los píxeles que quedan por debajo de 0 se ven negros.


In [ ]:
output = F.conv2d(image_batch, identity, bias=torch.tensor([-100.0]))  # un bias por filtro de salida
show_tensor_image(output, title="bias=-100", vmin=0, vmax=255)

print(f"Rango de la salida: [{output.min().item():.0f}, {output.max().item():.0f}]")


### Lectura opcional: convoluciones con varios canales

> **Esta sección es lectura opcional.** La clase continúa en **Pooling**. Conviene leerla antes de la próxima clase, donde las imágenes de CIFAR-10 tienen 3 canales (RGB).

Hasta acá la entrada tuvo un solo canal. ¿Qué pasa con una imagen a color (3 canales) o con la salida de una capa anterior que tiene, por ejemplo, 6 feature maps?

![Image](https://d2l.ai/_images/conv-multi-in.svg)

Cuando la entrada tiene $C_{in}$ canales, **cada filtro también tiene $C_{in}$ canales**: es un tensor de $C_{in} \times k \times k$. Se hace la correlación de cada canal del filtro con el canal correspondiente de la entrada y **se suman los resultados**: un filtro produce **un solo** feature map de salida, sin importar cuántos canales tenga la entrada. Si queremos $C_{out}$ feature maps, necesitamos $C_{out}$ filtros.

Por eso el tensor de pesos de una convolución tiene forma `[out_channels, in_channels, kernel_h, kernel_w]`, como decía el comentario del primer ejemplo. Y por eso, al apilar capas, el `out_channels` de una capa es el `in_channels` de la siguiente.

Probemos con la misma imagen pero en RGB:


In [ ]:
image_rgb = read_image(str(Path("assets") / "PrisonMike.png"), ImageReadMode.RGB)
print("Forma de la imagen RGB:", image_rgb.shape)  # [3, H, W]

image_rgb_batch = image_rgb.to(torch.float32).unsqueeze(0)  # [1, 3, H, W]

# un filtro con 3 canales de entrada: aplica `edge` a R, G y B y suma los tres resultados
# (dividimos por 3 para que la suma de los tres canales quede en la misma escala que un canal)
edge_rgb = edge.repeat(1, 3, 1, 1) / 3  # [1, 1, 3, 3] -> [1, 3, 3, 3]
print("Forma del filtro:", edge_rgb.shape)

output = F.conv2d(image_rgb_batch, edge_rgb)
print("Forma de la salida:", output.shape)  # 1 filtro -> 1 canal de salida

show_tensor_images(
    [image_rgb, output[0].abs()],
    titles=["RGB (3 canales)", "Salida de 1 filtro (1 canal)"],
    figsize=(12, 5),
    vmin=0,
    vmax=255,
)


Con esto ya podemos calcular cuántos **parámetros entrenables** tiene una capa convolucional:

$$ \text{Parámetros} = (K_h \times K_w \times C_{\text{in}} + 1) \times C_{\text{out}} $$

Donde:
- $K_h$, $K_w$ son el alto y el ancho del filtro.
- $C_{\text{in}}$ es el número de canales de entrada.
- $C_{\text{out}}$ es el número de filtros (o canales de salida).
- El término $+1$ es el sesgo (bias) de cada filtro.

Notar que **la cantidad de parámetros no depende del tamaño de la imagen**: un filtro 3×3 tiene 9 pesos tanto para una imagen de 28×28 como para una de 1000×1000. Ese es el efecto de compartir pesos. En una `nn.Linear`, en cambio, los parámetros crecen con la cantidad de entradas.

> El sesgo es opcional pero está activado por defecto en PyTorch. Se puede desactivar con `bias=False`, y la fórmula se simplifica a $K_h \times K_w \times C_{\text{in}} \times C_{\text{out}}$.

> **Convolución 1×1**: un caso particular útil es `kernel_size=1`. No mira vecinos: para cada píxel combina linealmente los $C_{in}$ canales y devuelve $C_{out}$ canales. Se usa para aumentar o reducir la cantidad de canales sin tocar el alto y el ancho. La van a ver en la próxima clase (DenseNet).


In [ ]:
conv = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=3)

print("Forma de conv.weight:", conv.weight.shape)  # [out_channels, in_channels, kh, kw]
print("Forma de conv.bias:  ", conv.bias.shape)  # [out_channels]

n_params = sum(p.numel() for p in conv.parameters())
print(f"Parámetros: {n_params} = (3 * 3 * 3 + 1) * 6 = {(3 * 3 * 3 + 1) * 6}")


### Pooling

El **Pooling** es una operación clave en las redes neuronales convolucionales (CNNs) que se utiliza para reducir las dimensiones espaciales (ancho y alto) de los mapas de características. Esto se logra resumiendo las características en regiones específicas de la imagen de entrada. El pooling ayuda a disminuir la cantidad de parámetros de las capas siguientes y el costo computacional, y también proporciona cierta invariancia a las traslaciones: si un borde se corre un píxel dentro de la ventana, el máximo de la ventana no cambia.

> Cabe destacar que el pooling no tiene parámetros entrenables, ya que simplemente aplica una función fija a una región de la imagen. Se aplica **canal por canal**, de forma independiente: no mezcla canales ni cambia su cantidad.

#### Tipos de Pooling

Existen varios tipos de pooling, pero los más comunes son:

1. **Max Pooling**: Selecciona el valor máximo en cada ventana de pooling. Es útil para resaltar las características más prominentes.
2. **Average Pooling**: Calcula el promedio de los valores en cada ventana de pooling. Suaviza las características y es menos agresivo que el max pooling.

En PyTorch, podemos aplicar pooling utilizando la función [`torch.nn.functional.max_pool2d`](https://pytorch.org/docs/stable/generated/torch.nn.functional.max_pool2d.html) o [`torch.nn.functional.avg_pool2d`](https://pytorch.org/docs/stable/generated/torch.nn.functional.avg_pool2d.html).

#### Dimensiones de salida

Las dimensiones de salida del pooling se calculan con la misma fórmula que la convolución:

$$
\text{output\_size} = \left\lfloor \frac{\text{input\_size} - \text{kernel\_size} + 2 \times \text{padding}}{\text{stride}} \right\rfloor + 1
$$

> **Atención:** en el pooling, el `stride` por defecto es **igual a `kernel_size`** (las ventanas no se solapan), a diferencia de la convolución, donde el `stride` por defecto es 1. Por eso `max_pool2d(x, kernel_size=2)` reduce la imagen a la mitad, y `kernel_size=3` a un tercio.

Empecemos con un ejemplo numérico de 4×4 con ventana 2×2:


In [ ]:
x = torch.tensor(
    [[1, 3, 2, 0], [4, 8, 1, 1], [0, 2, 9, 5], [1, 1, 3, 7]], dtype=torch.float32
).reshape(1, 1, 4, 4)

print("Entrada:\n", x[0, 0])
print("\nMax pooling 2x2 (se queda con el máximo de cada ventana):\n", F.max_pool2d(x, kernel_size=2)[0, 0])
print("\nAvg pooling 2x2 (promedio de cada ventana):\n", F.avg_pool2d(x, kernel_size=2)[0, 0])


Ahora sobre la imagen. Además de la imagen original, aplicamos el pooling sobre la salida del filtro `edge`: ahí se ve la diferencia entre los dos tipos. El max pooling conserva los bordes (los valores altos sobreviven); el average pooling los diluye.


In [ ]:
# vemos un pooling en acción
max_pool_out = F.max_pool2d(image_batch, kernel_size=3)  # stride=3 por defecto
avg_pool_out = F.avg_pool2d(image_batch, kernel_size=3)

show_tensor_images(
    [image_f, max_pool_out, avg_pool_out],
    titles=["Original", "Max Pooling", "Avg Pooling"],
    vmin=0,
    vmax=255,
)

print(f"Shape de la imagen original: {image_batch.shape}")
print(f"Shape de la imagen con Max Pooling: {max_pool_out.shape}")  # floor((320 - 3) / 3) + 1 = 106
print(f"Shape de la imagen con Avg Pooling: {avg_pool_out.shape}")

# sobre la salida del detector de bordes
edge_map = F.conv2d(image_batch, edge).abs()
show_tensor_images(
    [edge_map, F.max_pool2d(edge_map, kernel_size=3), F.avg_pool2d(edge_map, kernel_size=3)],
    titles=["|Edge|", "Max Pooling de |Edge|", "Avg Pooling de |Edge|"],
    vmin=0,
    vmax=255,
)


> Un caso especial es el **pooling global** ([`nn.AdaptiveAvgPool2d(1)`](https://pytorch.org/docs/stable/generated/torch.nn.AdaptiveAvgPool2d.html)): promedia cada feature map completo y lo reduce a un único número. Las arquitecturas modernas lo usan al final de la red, en lugar de `flatten`, para no depender del tamaño de la imagen de entrada. Lo van a ver en DenseNet.


## Carga de datos

### Dataset FashionMNIST

El dataset FashionMNIST es una alternativa al clásico MNIST. En lugar de contener dígitos escritos a mano, FashionMNIST incluye imágenes de artículos de moda divididos en 10 clases, tales como camisetas, zapatos y bolsos. Cada imagen es de 28x28 píxeles en escala de grises, lo que hace que este dataset sea ideal para probar algoritmos de clasificación de imágenes. Tiene 60 000 imágenes de entrenamiento y 10 000 de prueba, con exactamente 6 000 y 1 000 imágenes por clase respectivamente (está perfectamente balanceado).

Para facilitar el manejo del dataset FashionMNIST, utilizaremos la biblioteca `torchvision.datasets`, la cual proporciona una manera sencilla de cargar y preprocesar estos datos.

**Cargando el dataset con torchvision**

La clase `torchvision.datasets.FashionMNIST` permite descargar y cargar el dataset FashionMNIST de manera eficiente.

```python
datasets.FashionMNIST(
    root="data",
    train=True,
    transform=None,
    target_transform=None,
    download=True
)
```

Algunos de los parámetros más importantes son:

- `root`: Directorio donde se almacenarán los datos.
- `train`: Si es `True`, carga el conjunto de entrenamiento; si es `False`, carga el conjunto de prueba.
- `transform`: Transformaciones que se aplicarán a los datos.
- `target_transform`: Transformaciones que se aplicarán a las etiquetas.
- `download`: Si es `True`, descarga el dataset desde internet y lo almacena en `root`

Existen otros datasets disponibles en `torchvision.datasets`, como CIFAR10, CIFAR100, MNIST, etc. Puedes consultar la [documentación oficial](https://pytorch.org/vision/stable/datasets.html) para más información.


#### Transformaciones

Las transformaciones son operaciones que se aplican a cada muestra al momento de leerla del dataset, antes de que llegue a la red neuronal. Sin transformaciones, `FashionMNIST` devuelve imágenes PIL (una biblioteca de imágenes de Python), no tensores. Usamos `T.Compose` para encadenar varias transformaciones:

1. [`T.ToImage`](https://pytorch.org/vision/stable/generated/torchvision.transforms.v2.ToImage.html): Convierte la imagen PIL a un tensor de imagen `[C, H, W]` de tipo `uint8` (valores de 0 a 255).
2. [`T.ToDtype`](https://pytorch.org/vision/stable/generated/torchvision.transforms.v2.ToDtype.html): Convierte los datos a un tipo de dato específico. Con `scale=True`, además divide por 255 y deja los valores en el rango $[0, 1]$. **Sin `scale=True` la red recibiría valores entre 0 y 255**, mucho más grandes que los pesos iniciales, y el entrenamiento sería más lento e inestable.
3. [`T.Normalize`](https://pytorch.org/vision/stable/generated/torchvision.transforms.v2.Normalize.html): Estandariza cada canal: resta la media y divide por el desvío estándar. Es la misma estandarización que hicimos en la clase 02 con las variables de California Housing (LeCun, *Efficient BackProp*), y por el mismo motivo se calcula **solo con los datos de entrenamiento**. Para FashionMNIST, la media y el desvío de los píxeles de train (ya escalados a $[0, 1]$) son $0.2860$ y $0.3530$.

Estas transformaciones se aplican automáticamente cuando cargamos el dataset utilizando la clase `datasets.FashionMNIST` y se especifican en el parámetro `transform`. Para ver más detalles sobre las transformaciones disponibles en PyTorch, puedes consultar la [documentación oficial](https://pytorch.org/vision/stable/auto_examples/transforms/plot_transforms_getting_started.html).

> Algunas transformaciones pueden usarse como **data augmentation**, es decir, para aumentar la variedad de los datos de entrenamiento: rotar, recortar, cambiar el brillo, etc. Lo vamos a ver en la próxima clase.


In [ ]:
DATA_DIR = Path("data")

# media y desvío de los píxeles de train (escalados a [0, 1]); un valor por canal
FMNIST_MEAN = [0.2860]
FMNIST_STD = [0.3530]

transforms = T.Compose(
    [
        # TODO: en orden,
        #   1. convertir la imagen PIL a tensor (T.ToImage)
        #   2. pasar a float32 escalando a [0, 1] (T.ToDtype con scale=True)
        #   3. estandarizar con FMNIST_MEAN y FMNIST_STD (T.Normalize)
    ]
)

fmnist_train_dataset = datasets.FashionMNIST(
    DATA_DIR, download=True, train=True, transform=transforms
)

fmnist_test_dataset = datasets.FashionMNIST(
    DATA_DIR, download=True, train=False, transform=transforms
)


In [ ]:
name_classes = fmnist_train_dataset.classes
nclasses = len(name_classes)

print(f"Clases: {name_classes}")
print(f"Train: {len(fmnist_train_dataset)} imágenes | Test: {len(fmnist_test_dataset)} imágenes")


Separamos una parte del conjunto de entrenamiento (20%) para usarla como conjunto de validación, igual que en la clase 02. Como FashionMNIST está balanceado, no hace falta un split estratificado como el de la clase 03: un split aleatorio simple deja las clases repartidas de forma pareja.

> Usamos un `Generator` con la semilla para que el split sea siempre el mismo, sin depender de cuántos números aleatorios se consumieron antes en la notebook.


In [ ]:
fmnist_train_dataset, fmnist_val_dataset = random_split(
    fmnist_train_dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(SEED)
)

print(f"Train: {len(fmnist_train_dataset)} | Val: {len(fmnist_val_dataset)} | Test: {len(fmnist_test_dataset)}")


### DataLoaders


In [ ]:
def get_dataloaders(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS):
    train_loader = DataLoader(
        fmnist_train_dataset,
        batch_size=batch_size,
        shuffle=True,  # solo en train: en val y test el orden no importa y así la evaluación es reproducible
        num_workers=num_workers,
    )

    val_loader = DataLoader(
        fmnist_val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    test_loader = DataLoader(
        fmnist_test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = get_dataloaders()


Antes de definir el modelo, miremos un batch. Nos interesa el `shape` (el DataLoader ya nos entrega la dimensión de canal, `[N, 1, 28, 28]`), el tipo de dato y el rango de valores: si las transformaciones están bien, los píxeles quedan centrados en 0 con desvío 1 (el fondo negro vale $(0 - 0.286) / 0.353 \approx -0.81$).


In [ ]:
x_batch, y_batch = next(iter(train_loader))

print(f"x: {x_batch.shape} {x_batch.dtype} | rango: [{x_batch.min().item():.2f}, {x_batch.max().item():.2f}]")
print(f"y: {y_batch.shape} {y_batch.dtype} | primeras etiquetas: {y_batch[:8].tolist()}")

show_tensor_images(
    [x_batch[i] for i in range(8)],
    titles=[name_classes[y_batch[i]] for i in range(8)],
    figsize=(16, 3),
)


## Modelo de CNN

Igual que con el modelo FeedForward, para crear un modelo usando convoluciones necesitamos crear una clase, definir los métodos `__init__` y `forward` y especificar la arquitectura y el comportamiento de los componentes del modelo.

En particular vamos a usar:

- [Capas convolucionales de 2D](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html#torch.nn.Conv2d) a las que tenemos que especificarles la cantidad de canales de entrada (`in_channels`: 1 para gris, 3 para color, o la cantidad de filtros de la capa anterior), la cantidad de filtros a usar (`out_channels`), el tamaño de los mismos (`kernel_size`) y si aplicamos `padding` o no (esto nos permite hacer convoluciones que no modifiquen el tamaño de las imágenes).

- [Capas de average pooling](https://pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html#torch.nn.AvgPool2d) o [max pooling](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html#torch.nn.MaxPool2d) a las que tenemos que decirles el tamaño de la ventana (`kernel_size`) y el paso (`stride`).

- Finalmente, `flatten` y capas lineales como hicimos anteriormente.

### LeNet-5

Vamos a implementar [LeNet-5](https://d2l.ai/chapter_convolutional-neural-networks/lenet.html), una de las primeras redes neuronales convolucionales que se utilizó en la práctica. Fue propuesta por Yann LeCun y colaboradores en 1998 ([*Gradient-Based Learning Applied to Document Recognition*](http://vision.stanford.edu/cs598_spring07/papers/Lecun98.pdf)) para el reconocimiento de dígitos escritos a mano en cheques bancarios. La arquitectura es la siguiente:

![Image](./assets/LeNet_architecture.png)

Capa por capa, con la entrada de 28×28 que usamos nosotros:

| Capa | Operación | Salida `[C, H, W]` | Cómo se calcula |
|---|---|---|---|
| entrada | — | `[1, 28, 28]` | |
| C1 | `Conv2d(1, 6, kernel_size=5, padding=2)` + tanh | `[6, 28, 28]` | $(28 - 5 + 2 \cdot 2) / 1 + 1 = 28$ |
| S2 | `AvgPool2d(kernel_size=2, stride=2)` | `[6, 14, 14]` | $28 / 2$ |
| C3 | `Conv2d(6, 16, kernel_size=5)` + tanh | `[16, 10, 10]` | $14 - 5 + 1 = 10$ |
| S4 | `AvgPool2d(kernel_size=2, stride=2)` | `[16, 5, 5]` | $10 / 2$ |
| C5 | `Conv2d(16, 120, kernel_size=5)` + tanh | `[120, 1, 1]` | $5 - 5 + 1 = 1$ |
| flatten | `x.flatten(start_dim=1)` | `[120]` | $120 \cdot 1 \cdot 1$ |
| F6 | `Linear(120, 84)` + tanh | `[84]` | |
| salida | `Linear(84, 10)` | `[10]` | un logit por clase |

Algunas notas sobre el diagrama y la tabla:

- **28×28 vs 32×32.** En la arquitectura original la entrada es de 32×32 píxeles. Nuestras imágenes son de 28×28, y sin cambios la red rompería: 28 → 24 → 12 → 8 → 4, y C5 no puede aplicar un kernel de 5×5 sobre 4×4. Con `padding=2` en C1 la salida de C1 es 28×28, y de ahí en adelante los tamaños coinciden exactamente con el diagrama.
- **C5 es una convolución**, no una capa lineal: aplica 120 filtros de 5×5 sobre un mapa de 5×5, así que cada filtro produce un único número. La salida `[120, 1, 1]` se aplana a un vector de 120, que es el `in_features` de F6. Si cambian la arquitectura (por ejemplo, agregando padding o quitando un pooling), el tamaño de entrada de la primera `Linear` cambia y hay que recalcularlo con la fórmula.
- **Pooling.** El diagrama dice "MaxP", pero LeNet-5 original usa *subsampling* (promedio), y nosotros también: `AvgPool2d`.
- **Activación.** LeNet-5 usa `tanh` (en 1998 ReLU todavía no se usaba). Hoy se usa ReLU casi siempre; probar el cambio es el ejercicio 1. La capa de salida **no lleva activación**: `CrossEntropyLoss` aplica la softmax internamente, como en la clase 03.

> Referencia adicional con el conteo de parámetros por capa: [LeNet-5 architecture (GeeksforGeeks)](https://www.geeksforgeeks.org/lenet-5-architecture/).


In [ ]:
# Salida esperada de summary (usarla para verificar la implementación):
#
# LeNet                                    [128, 10]                 --
# ├─Conv2d: 1-1                            [128, 6, 28, 28]          156
# ├─AvgPool2d: 1-2                         [128, 6, 14, 14]          --
# ├─Conv2d: 1-3                            [128, 16, 10, 10]         2,416
# ├─AvgPool2d: 1-4                         [128, 16, 5, 5]           --
# ├─Conv2d: 1-5                            [128, 120, 1, 1]          48,120
# ├─Linear: 1-6                            [128, 84]                 10,164
# ├─Linear: 1-7                            [128, 10]                 850
# Total params: 61,706


class LeNet(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(LeNet, self).__init__()  # obligatorio: inicializa nn.Module para que registre capas y parámetros
        # TODO: definir las capas como atributos siguiendo la tabla: c1, s2, c3, s4, c5, f6, output.
        #       c1 lleva padding=2; las capas de pooling son nn.AvgPool2d(kernel_size=2, stride=2)
        pass

    def forward(self, x):
        # TODO: aplicar las capas en orden con F.tanh después de c1, c3, c5 y f6.
        #       Entre c5 y f6 aplanar con x.flatten(start_dim=1). La salida no lleva activación
        pass


summary(LeNet(1, nclasses), input_size=(BATCH_SIZE, 1, 28, 28))


Verifiquemos la columna de parámetros con la fórmula $(K_h \times K_w \times C_{in} + 1) \times C_{out}$ (se explica en la sección opcional de canales; en resumen: cada filtro tiene un peso por cada posición del kernel y por cada canal de entrada, más un sesgo):

- C1: $(5 \cdot 5 \cdot 1 + 1) \cdot 6 = 156$
- C3: $(5 \cdot 5 \cdot 6 + 1) \cdot 16 = 2\,416$
- C5: $(5 \cdot 5 \cdot 16 + 1) \cdot 120 = 48\,120$
- F6: $120 \cdot 84 + 84 = 10\,164$ (una `Linear` común)
- salida: $84 \cdot 10 + 10 = 850$

Total: $61\,706$. Las tres convoluciones juntas tienen menos parámetros que una sola capa `Linear(784, 84)` ($65\,940$).


### Entrenamiento

A partir de esta clase, las funciones `train`, `evaluate` y `EarlyStopping` que implementaron en la clase 03 viven en `utils.py`, para no repetirlas en cada notebook. La función `train` es la versión con early stopping:

```python
train(model, optimizer, criterion, train_loader, val_loader, device,
      do_early_stopping=True, patience=5, epochs=10, log_fn=print_log, log_every=1)
```

Devuelve dos listas con la pérdida de entrenamiento y de validación por época, que podemos graficar con `plot_training`.


In [ ]:
# definicion de hiperparametros
LR = 0.001
EPOCHS = 10

# TODO:
#   1. instanciar el modelo en DEVICE con el nombre lenet_model (se usa en las celdas siguientes)
#   2. optimizador Adam con lr=LR y criterio nn.CrossEntropyLoss()
#   3. entrenar con train(...) usando do_early_stopping=True y patience=3;
#      guardar las dos listas que devuelve en train_errors y val_errors
#   4. graficar con plot_training(train_errors, val_errors)


## Resultados

Evaluamos en el conjunto de **test**, que el modelo nunca vio: el de validación ya lo usamos para decidir cuándo frenar el entrenamiento (early stopping), así que su resultado es levemente optimista. Es el mismo criterio que en las clases 02 y 03.


In [ ]:
model_classification_report(lenet_model, test_loader, DEVICE, nclasses, target_names=name_classes)


Mirando el reporte por clase: las clases con peor F1 son prendas superiores (*Shirt*, *T-shirt/top*, *Pullover*, *Coat*), que en 28×28 píxeles son difíciles de distinguir incluso para una persona. Las clases con formas distintivas (*Trouser*, *Bag*, *Sneaker*, *Ankle boot*) superan el 95%.


## Lectura opcional: ¿qué aprendió la red?

> **Esta sección es lectura opcional.** Requiere haber entrenado `lenet_model`. Los ejercicios están al final de la notebook.

Al principio dijimos que `nn.Conv2d` aprende los filtros. Veámoslos. Los pesos de la primera capa tienen forma `[6, 1, 5, 5]`: 6 filtros de 5×5 sobre 1 canal, así que podemos dibujarlos como imágenes chicas. Los filtros de la primera capa suelen parecerse a detectores de bordes y de contraste como los que definimos a mano; los de capas más profundas (C3 tiene 6 canales de entrada, C5 tiene 16) ya no se pueden dibujar de forma tan directa.


In [ ]:
c1_weights = lenet_model.c1.weight.detach().cpu()  # [6, 1, 5, 5]
print("Forma de los pesos de C1:", c1_weights.shape)

show_tensor_images(
    [c1_weights[i] for i in range(c1_weights.shape[0])],
    titles=[f"Filtro {i}" for i in range(c1_weights.shape[0])],
    figsize=(15, 3),
)


Más informativo que los filtros en sí es ver **qué producen** sobre una imagen: los feature maps. Tomamos una imagen de test y la pasamos capa por capa. Después de C1 tenemos 6 mapas de 28×28 (cada filtro responde a un patrón distinto); después de S2 los mismos 6 mapas, reducidos a 14×14.


In [ ]:
x_test, y_test = next(iter(test_loader))
img = x_test[:1].to(DEVICE)  # una sola imagen, con dimensión de batch: [1, 1, 28, 28]

lenet_model.eval()
with torch.no_grad():
    c1_out = F.tanh(lenet_model.c1(img))  # [1, 6, 28, 28]
    s2_out = lenet_model.s2(c1_out)  # [1, 6, 14, 14]

print("Salida de C1:", c1_out.shape, "| Salida de S2:", s2_out.shape)

show_tensor_image(x_test[0], title=f"Entrada: {name_classes[y_test[0]]}")
show_tensor_images(
    [c1_out[0, i].unsqueeze(0).cpu() for i in range(6)],
    titles=[f"C1 - mapa {i}" for i in range(6)],
    figsize=(15, 3),
)
show_tensor_images(
    [s2_out[0, i].unsqueeze(0).cpu() for i in range(6)],
    titles=[f"S2 - mapa {i}" for i in range(6)],
    figsize=(15, 3),
)


Por último, algunas predicciones sobre test, y en particular las que la red se equivoca: ayuda a entender *qué* confunde el modelo, no solo cuánto.


In [ ]:
with torch.no_grad():
    preds = torch.argmax(lenet_model(x_test.to(DEVICE)), dim=1).cpu()

# primeras 8 imágenes del batch de test
show_tensor_images(
    [x_test[i] for i in range(8)],
    titles=[f"pred: {name_classes[preds[i]]}\nreal: {name_classes[y_test[i]]}" for i in range(8)],
    figsize=(16, 3.5),
)

# hasta 8 imágenes mal clasificadas
wrong = torch.nonzero(preds != y_test).flatten()[:8]
print(f"Mal clasificadas en este batch: {(preds != y_test).sum().item()} de {len(y_test)}")
show_tensor_images(
    [x_test[i] for i in wrong],
    titles=[f"pred: {name_classes[preds[i]]}\nreal: {name_classes[y_test[i]]}" for i in wrong],
    figsize=(16, 3.5),
)


## Ejercicios

1. **LeNet moderna**: Reemplazar `tanh` por ReLU y `AvgPool2d` por `MaxPool2d`. ¿Cambia el rendimiento? ¿Y la velocidad de convergencia?
2. **CNN vs MLP**: Entrenar el MLP de la clase 03 sobre las imágenes aplanadas (agregar `nn.Flatten()` como primera capa, `input_size=784`). Comparar cantidad de parámetros (`summary`) y accuracy en test con LeNet.
3. **Sin normalizar**: Quitar `T.Normalize` (y luego también `scale=True`) de las transformaciones y volver a entrenar. Observar las curvas de pérdida. ¿Por qué pasa lo que pasa?
4. **Implementar una CNN más profunda**: Modificar la arquitectura para agregar más capas convolucionales y/o capas totalmente conectadas. Usar `padding` para que las convoluciones no reduzcan el tamaño y poder apilar varias antes de cada pooling. ¿Cómo afecta esto al rendimiento del modelo? Recordar recalcular el `in_features` de la primera `Linear`.
5. **Weights & Biases**: Utilizar la librería [Weights & Biases](https://wandb.ai/site) para correr experimentos y comparar diferentes hiperparámetros (learning rate, batch size, activación, cantidad de filtros).


#### Lectura adicional

- [Dive into Deep Learning, capítulo 7](https://d2l.ai/chapter_convolutional-neural-networks/index.html): de capas densas a convoluciones, padding y stride, canales, pooling y LeNet. Es la fuente de las figuras de esta notebook.
- [CNN Explainer](https://poloclub.github.io/cnn-explainer/): visualización interactiva de una CNN completa, capa por capa, sobre imágenes reales.
- [Setosa – Image Kernels](https://setosa.io/ev/image-kernels/): probar filtros a mano sobre una imagen.
- [3Blue1Brown – But what is a convolution?](https://www.youtube.com/watch?v=KuXjwB4LzSA): la intuición de la convolución, desde probabilidades hasta imágenes.
- Dumoulin & Visin, [*A guide to convolution arithmetic for deep learning*](https://arxiv.org/abs/1603.07285) y su [repo con animaciones](https://github.com/vdumoulin/conv_arithmetic) de padding y stride.
- LeCun, Bottou, Bengio & Haffner (1998), [*Gradient-Based Learning Applied to Document Recognition*](http://vision.stanford.edu/cs598_spring07/papers/Lecun98.pdf): el paper de LeNet-5.
- Zeiler & Fergus (2013), [*Visualizing and Understanding Convolutional Networks*](https://arxiv.org/abs/1311.2901): qué aprenden los filtros de cada capa.
